# Clustering Semantico dei Report di Insicurezza Alimentare (IPC)

Questo notebook implementa una pipeline avanzata per il caricamento, la pulizia, l'embedding denso e il clustering semantico dei report IPC.  
L'obiettivo è raggruppare le situazioni di crisi in base alla tipologia di insicurezza alimentare e ai driver che le causano (es. conflitti, siccità, crisi economiche), indipendentemente dal paese.

La pipeline è progettata per essere eseguita sulla **GPU di Google Colab** e include:
1. Setup dell'ambiente (Montaggio Drive e installazione delle librerie).
2. Caricamento e pulizia dei testi (rimozione dell'intestazione metadati).
3. Generazione di embedding con **Instructor-Large** (`hkunlp/instructor-large`), un modello avanzato esplicitamente ottimizzato per seguire istruzioni arbitrarie per compiti come il clustering.
4. Riduzione dimensionale con **UMAP** (5D per clustering, 2D per visualizzazione).
5. Clustering con **K-Means** (e scelta ottimale di K con metriche geometriche come *Silhouette Score* e *Davies-Bouldin Index*) e **HDBSCAN**.
6. **Validazione della Coerenza Semantica**:
   - **Lemmatizzazione (NLTK)** e rimozione dei nomi dei paesi nel TF-IDF per parole chiave pulite.
   - **Tabella di Contingenza** (Paese vs. Cluster) ed entropia geografica per escludere il bias geografico.
   - **LLM-as-a-Judge** (tramite API Mistral) per descrivere i cluster e valutarne la coerenza semantica.
7. Visualizzazione interattiva 2D con **Plotly**.

## 1. Setup dell'Ambiente e Importazione Librerie

In [4]:
import pandas as pd
df = pd.read_csv("df_duplicati.csv")
df

,Unnamed: 0,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,gruppo,Llama_3_2_3B_anonimo,Llama_3_1_8B_anonimo,Llama_3_1_8B_8bit_anonimo
0,7,El_Salvador_Mar_2025_-_Feb_2026_KeyResults.txt,El Salvador,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
1,9,Honduras_Mar_2025_-_Feb_2026_KeyResults.txt,Honduras,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
2,23,Guatemala_Mar_2025_-_Feb_2026_KeyResults.txt,Guatemala,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]: \nThe affected areas are...
3,31,El_Salvador_Jun_2020_-_Aug_2020_KeyResults.txt,El Salvador,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima..."
4,35,Honduras_Jun_2020_-_Aug_2020_KeyResults.txt,Honduras,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...
5,59,Guatemala_Jun_2020_-_Aug_2020_KeyResults.txt,Guatemala,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima..."
6,68,El_Salvador_Jun_2022_-_Aug_2022_KeyResults.txt,El Salvador,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
7,70,Honduras_Jun_2022_-_Aug_2022_KeyResults.txt,Honduras,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
8,74,Guatemala_Jun_2022_-_Aug_2022_KeyResults.txt,Guatemala,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...
9,162,El_Salvador_Nov_2018_-_Apr_2019_KeyResults.txt,El Salvador,Nov 2018 / Apr 2019,Nov 2018,Apr 2019,The Tri-national Border Federation of Río Lemp...,3.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...


In [5]:
# 1.3 Import delle librerie necessarie
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import pairwise_distances_argmin_min
import plotly.express as px
import torch
import scipy.stats as stats
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Impostiamo il seed per la riproducibilità dei risultati
np.random.seed(42)

# Download delle risorse NLTK per la lemmatizzazione
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet = True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)
print("NLTK e librerie importate correttamente.")

NLTK e librerie importate correttamente.


## 3. Generazione degli Embedding con Modello Instructor-Large (GPU)

Utilizziamo il modello **`hkunlp/instructor-large`**. Questo modello è stato esplicitamente addestrato per seguire istruzioni di testo arbitrarie per compiti come il clustering, formattate come coppie di `[istruzione, testo]`. Questo ci consente di guidare lo spazio semantico a focalizzarsi sui driver delle crisi e sulle condizioni di insicurezza alimentare.

In [6]:
df.columns

Index(['Unnamed: 0', 'nome_file', 'paese', 'periodo', 'inizio_periodo',
       'fine_periodo', 'testo_originale', 'gruppo', 'Llama_3_2_3B_anonimo',
       'Llama_3_1_8B_anonimo', 'Llama_3_1_8B_8bit_anonimo'],
      dtype='object')

In [7]:
# 1.1 Monta Google Drive per accedere ai file dei report
#from google.colab import drive
import os
import pandas as pd

DRIVE_PATH = "./"
colonne_target = ['Llama_3_2_3B_anonimo', 'Llama_3_1_8B_anonimo', 'Llama_3_1_8B_8bit_anonimo']
for colonna_target in colonne_target:
    TESTO_DIR = df[colonna_target]


    # 3.1 Caricamento del modello Instructor-Large
    # 3.1 Caricamento del modello GTE-Large
    import torch
    from sentence_transformers import SentenceTransformer
    import einops

    # 1. Rilevamento corretto dell'acceleratore Apple Silicon (Metal Performance Shaders)
    MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Caricamento modello {MODEL_NAME} su dispositivo: {device}...")

    # 2. Inizializzazione con disattivazione del padding dinamico incompatibile con macOS
    model = SentenceTransformer(
        MODEL_NAME,
        device=device,
        trust_remote_code=True,
    )

    # Definiamo l'istruzione per orientare il clustering esplicitamente sui driver
    istruzione = "search_document: "

    # Il modello GTE richiede la concatenazione diretta dell'istruzione e del testo
    testi_con_istruzione = [istruzione + t for t in df[colonna_target]]

    print("Generazione degli embedding in corso...")
    embeddings = model.encode(testi_con_istruzione, normalize_embeddings=True, show_progress_bar=True)
    print(f"Embedding generati! Shape: {embeddings.shape}")

    # Salva gli embedding su Drive per riutilizzarli senza rigenerarli
    np.save(os.path.join(DRIVE_PATH, "embeddings_report_anonimi.npy"), embeddings)
    df.to_parquet(os.path.join(DRIVE_PATH, "metadata_report_anonimi.parquet"))
    print("Embedding e metadati salvati correttamente")

    # 4.1 Riduzione dimensionale con UMAP
    print("UMAP a 5 dimensioni per il clustering...")
    reducer_clustering = umap.UMAP(
        n_neighbors=15,
        min_dist=0.0,
        n_components=5,
        metric="cosine",
        random_state=42
    )
    embeddings_5d = reducer_clustering.fit_transform(embeddings)

    print("UMAP a 2 dimensioni per la visualizzazione...")
    reducer_visualization = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        metric="cosine",
        random_state=42
    )
    embeddings_2d = reducer_visualization.fit_transform(embeddings)

    # Inserimento coordinate UMAP nel dataframe
    df[f"x_{colonna_target}"] = embeddings_2d[:, 0]
    df[f"y_{colonna_target}"] = embeddings_2d[:, 1]

    # Salva gli embedding ridotti
    np.save(os.path.join(DRIVE_PATH, "embeddings_umap5d_report_anonimi.npy"), embeddings_5d)
    print("Riduzione dimensionale completata e salvata.")

    # 5.1.1 Valutazione geometrica su range di K
    inertie = []
    silhouette = []
    davies_bouldin = []
    K_range = range(2, 13)

    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(embeddings_5d)
        inertie.append(km.inertia_)
        silhouette.append(silhouette_score(embeddings_5d, labels))
        davies_bouldin.append(davies_bouldin_score(embeddings_5d, labels))


    # 5.2.1 Esecuzione HDBSCAN
    import hdbscan

    print("Calcolo dei cluster con HDBSCAN...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=5,
        metric="manhattan"
    )
    hdbscan_labels = clusterer.fit_predict(embeddings_5d)
    df[f"cluster_hdbscan_{colonna_target}"] = hdbscan_labels

    n_clusters_hdb = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
    n_noise_hdb = list(hdbscan_labels).count(-1)

Caricamento modello nomic-ai/nomic-embed-text-v1.5 su dispositivo: mps...


<All keys matched successfully>


Generazione degli embedding in corso...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.49it/s]
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Embedding generati! Shape: (35, 768)
Embedding e metadati salvati correttamente
UMAP a 5 dimensioni per il clustering...
UMAP a 2 dimensioni per la visualizzazione...
Riduzione dimensionale completata e salvata.
Calcolo dei cluster con HDBSCAN...
Caricamento modello nomic-ai/nomic-embed-text-v1.5 su dispositivo: mps...


...

<All keys matched successfully>


Generazione degli embedding in corso...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.55it/s]

Embedding generati! Shape: (35, 768)
Embedding e metadati salvati correttamente
UMAP a 5 dimensioni per il clustering...
UMAP a 2 dimensioni per la visualizzazione...
Riduzione dimensionale completata e salvata.
Calcolo dei cluster con HDBSCAN...



/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/envs/nlp_clustering/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in mat

In [16]:
# 1. Isola le colonne target per pulizia del codice
colonne_cluster = [
    'cluster_hdbscan_Llama_3_2_3B_anonimo',
    'cluster_hdbscan_Llama_3_1_8B_anonimo',
    'cluster_hdbscan_Llama_3_1_8B_8bit_anonimo'
]

# 2. Esegui l'aggregazione quantitativa (Consigliata per dataset estesi)
df_omogeneita = df.groupby('gruppo')[colonne_cluster].nunique()
df_omogeneita

,cluster_hdbscan_Llama_3_2_3B_anonimo,cluster_hdbscan_Llama_3_1_8B_anonimo,cluster_hdbscan_Llama_3_1_8B_8bit_anonimo
gruppo,,,
0.0,1,1,1
1.0,1,1,1
2.0,1,1,1
3.0,1,1,1
4.0,1,1,1
5.0,1,1,1
6.0,1,1,1
7.0,1,1,1
8.0,1,1,1


In [14]:
df

,Unnamed: 0,nome_file,paese,periodo,inizio_periodo,fine_periodo,testo_originale,gruppo,Llama_3_2_3B_anonimo,Llama_3_1_8B_anonimo,Llama_3_1_8B_8bit_anonimo,x_Llama_3_2_3B_anonimo,y_Llama_3_2_3B_anonimo,cluster_hdbscan_Llama_3_2_3B_anonimo,x_Llama_3_1_8B_anonimo,y_Llama_3_1_8B_anonimo,cluster_hdbscan_Llama_3_1_8B_anonimo,x_Llama_3_1_8B_8bit_anonimo,y_Llama_3_1_8B_8bit_anonimo,cluster_hdbscan_Llama_3_1_8B_8bit_anonimo
0,7,El_Salvador_Mar_2025_-_Feb_2026_KeyResults.txt,El Salvador,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,3.917336,-3.092032,-1,9.441715,10.410748,-1,-0.227834,0.174096,-1
1,9,Honduras_Mar_2025_-_Feb_2026_KeyResults.txt,Honduras,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,4.112453,-2.786137,-1,9.706458,10.707183,-1,0.066581,0.411465,-1
2,23,Guatemala_Mar_2025_-_Feb_2026_KeyResults.txt,Guatemala,Mar 2025 / Feb 2026,Mar 2025,Feb 2026,"During the current period (March to May 2025),...",0.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,[SHOCKS AND DRIVERS]: \nThe affected areas are...,3.591251,-2.895066,-1,9.169412,10.781019,-1,-0.015570,-0.002199,-1
3,31,El_Salvador_Jun_2020_-_Aug_2020_KeyResults.txt,El Salvador,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima...",5.537043,-3.007756,-1,8.240951,5.654067,-1,1.204649,1.661024,-1
4,35,Honduras_Jun_2020_-_Aug_2020_KeyResults.txt,Honduras,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]:\nThe affected areas are ...,5.504407,-2.785540,-1,8.011463,5.353070,-1,1.156667,2.146925,-1
5,59,Guatemala_Jun_2020_-_Aug_2020_KeyResults.txt,Guatemala,Jun 2020 / Aug 2020,Jun 2020,Aug 2020,"Overview\nFrom June to August 2020, the period...",1.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,"[SHOCKS AND DRIVERS]:\nEconomic factors, clima...",5.713109,-2.487803,-1,8.393112,5.451568,-1,1.431447,1.377671,-1
6,68,El_Salvador_Jun_2022_-_Aug_2022_KeyResults.txt,El Salvador,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,2.734009,-5.699531,-1,10.257873,10.286038,-1,0.897304,-0.299562,-1
7,70,Honduras_Jun_2022_-_Aug_2022_KeyResults.txt,Honduras,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,2.353041,-5.934326,-1,10.116608,9.866429,-1,0.764319,-0.924527,-1
8,74,Guatemala_Jun_2022_-_Aug_2022_KeyResults.txt,Guatemala,Jun 2022 / Aug 2022,Jun 2022,Aug 2022,Since several months have passed since the sec...,2.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,2.378244,-5.514226,-1,10.505216,10.086431,-1,1.290647,-0.529590,-1
9,162,El_Salvador_Nov_2018_-_Apr_2019_KeyResults.txt,El Salvador,Nov 2018 / Apr 2019,Nov 2018,Apr 2019,The Tri-national Border Federation of Río Lemp...,3.0,[SHOCKS AND DRIVERS]: \nThe underlying causes ...,[SHOCKS AND DRIVERS]: \nThe underlying causes 